# 04 — Clusterização de Processos por Assunto

**Objetivo:** Agrupar processos por similaridade textual das movimentações
para descobrir padrões temáticos não capturados pelos códigos de assunto.

**Pipeline:**
1. Carregar matriz TF-IDF (gerada no notebook 03)
2. Redução dimensional com SVD (LSA)
3. Escolha do número ideal de clusters (método do cotovelo + silhouette)
4. KMeans + interpretação dos clusters

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

from datajud.analysis.clustering import (
    clusterizar_processos,
    escolher_k,
    top_termos_por_cluster,
)

sns.set_theme(style='whitegrid')

## 1. Carregar artefatos do notebook 03

In [ ]:
matriz = sp.load_npz('../data/processed/tfidf_matrix.npz')
with open('../data/processed/tfidf_vetorizador.pkl', 'rb') as f:
    vetorizador = pickle.load(f)

df_mov = pd.read_parquet('../data/processed/movimentos_tjsp.parquet')

print(f'Matriz: {matriz.shape}')

## 2. Escolha do número de clusters

In [ ]:
df_k = escolher_k(matriz, k_min=2, k_max=12, plot=True)
df_k

## 3. Clusterização com K escolhido

In [ ]:
K_IDEAL = 6  # Ajuste conforme o gráfico acima

labels, modelo = clusterizar_processos(
    matriz,
    n_clusters=K_IDEAL,
    algoritmo='kmeans',
)

unique, counts = np.unique(labels, return_counts=True)
print('Distribuição dos clusters:')
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} processos')

## 4. Termos representativos por cluster

In [ ]:
# Agregar textos por processo (mesmo procedimento do notebook 03)
df_agg = (
    df_mov.groupby('numero_processo')['mov_nome']
    .apply(lambda x: ' '.join(x.dropna().astype(str)))
    .reset_index()
)

termos = top_termos_por_cluster(
    labels,
    vetorizador,
    df_agg,
    coluna_texto='mov_nome',
    top_n=10,
)

for cluster_id, palavras in termos.items():
    print(f'Cluster {cluster_id}: {" | ".join(palavras)}')

## 5. Visualização 2D (PCA / SVD)

In [ ]:
svd2 = TruncatedSVD(n_components=2, random_state=42)
X2 = normalize(svd2.fit_transform(matriz))

df_vis = pd.DataFrame(X2, columns=['dim1', 'dim2'])
df_vis['cluster'] = labels.astype(str)

fig, ax = plt.subplots(figsize=(9, 7))
sns.scatterplot(
    data=df_vis, x='dim1', y='dim2', hue='cluster',
    palette='tab10', alpha=0.6, s=50, ax=ax
)
ax.set_title(f'Clusters de Processos (K={K_IDEAL}) — SVD 2D')
plt.tight_layout()
plt.show()

## 6. Salvar resultado

In [ ]:
df_agg['cluster'] = labels
df_agg.to_parquet('../data/processed/processos_clusters.parquet', index=False)
print('Clusters salvos em data/processed/processos_clusters.parquet')